# DINOv2 50% frame selection + adaptive DivPrune at 15% visual-token retention

Run all cells in order with the current `benchmark-code` and `meld-dataset` Kaggle inputs, a GPU, and Internet. The experiment first retains 50% of decoded frames with the existing DINOv2-base centroid-seeded max-min cosine selector. It then applies DivPrune to Qwen's projected visual tokens **before decoder block 0**, keeping approximately 15% of the *unpruned Qwen visual-token count* per sample. All text and special tokens remain.

A fixed frame-ratio × token-ratio calculation would not reliably produce 15% Qwen tokens: Qwen can change visual tokens per frame after frame selection. This notebook uses `--target-combined-visual-retain-ratio 0.15`, which counts the unpruned processor's visual tokens and chooses the DivPrune count separately for each sample. Exact retention can vary by one token from rounding, or be lower if the selected frames contain fewer tokens than the target.

The notebook extracts train, dev, and test embeddings, trains a matched MLP, evaluates the frozen full-baseline MLP, validates actual retention, and compares predictions on shared sample IDs. Extraction saves progress every 25 samples. Reruns resume completed samples when the same `/kaggle/working` files are preserved or restored.

## 1. Inputs and output paths

In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
MELD_ROOT = Path("/kaggle/input/datasets/pushkarsingh2005/meld-dataset/MELD.Raw")
RUN_ROOT = Path("/kaggle/working/experiments/combined_pruning/qwen2_5_vl_3b_dinov2_050_divprune_target_015")
INDEX_DIR = RUN_ROOT / "data"
EMBEDDINGS = RUN_ROOT / "embeddings"
RESULTS = RUN_ROOT / "results"
MATCHED = RESULTS / "matched_mlp"
FIXED = RESULTS / "fixed_baseline_mlp"
EXTRACTOR = CODE / "scripts/extraction/extract_qwen_shared_embeddings.py"
PREPARE = CODE / "scripts/data/prepare_meld.py"
TRAIN = CODE / "scripts/classification/train_mlp.py"
EVALUATE = CODE / "scripts/classification/evaluate_mlp.py"
BASELINE = CODE / "experiments/baselines/qwen2_5_vl_3b_shared"
BASELINE_CHECKPOINT = BASELINE / "outputs/best_mlp.pt"
FRAME_RETAIN = 0.50
TARGET_VISUAL_RETAIN = 0.15

for path in (CODE, MELD_ROOT, EXTRACTOR, PREPARE, TRAIN, EVALUATE, BASELINE_CHECKPOINT,
             BASELINE / "outputs/dev_predictions.csv", BASELINE / "outputs/test_predictions.csv"):
    assert path.exists(), f"Missing {path}. Attach the current Kaggle inputs."
assert "--target-combined-visual-retain-ratio" in EXTRACTOR.read_text(), "Update benchmark-code: adaptive combined retention is missing."
assert 'INDEXER_IMPLEMENTATION = "directory_filename_listing_v1"' in PREPARE.read_text(), "Update the benchmark-code Kaggle input to the latest revision: the fast MELD indexer is missing."
for path in (INDEX_DIR, EMBEDDINGS, RESULTS):
    path.mkdir(parents=True, exist_ok=True)
print("Run directory:", RUN_ROOT)

## 2. Dependencies and GPU

In [ ]:
!python -m pip install -q "transformers==4.57.1" "qwen-vl-utils[decord]==0.0.14" accelerate decord pandas numpy scikit-learn matplotlib Pillow tqdm PyYAML opencv-python-headless

In [ ]:
import json
from importlib.metadata import version
import shutil
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from sklearn.metrics import accuracy_score, f1_score

# Extraction runs in fresh !python processes, so the kernel's cached import does not matter.
assert version("transformers") == "4.57.1", "The dependency install cell did not install Transformers 4.57.1."
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 2**30:.1f} GiB")

## 3. Prepare Kaggle-local MELD indexes

The indexes are rebuilt so video paths point to the attached dataset rather than a previous machine. The indexer prints progress for each split and lists video filenames once, avoiding thousands of slow file checks on Kaggle's mounted input.

In [ ]:
!python -u {PREPARE} \
  --meld-root {MELD_ROOT} \
  --split all \
  --output-dir {INDEX_DIR}

for split in ("train", "dev", "test"):
    index = pd.read_csv(INDEX_DIR / f"meld_{split}_index.csv")
    assert len(index) > 0 and index.sample_id.is_unique
    assert index.video_exists.eq(True).all()
    assert Path(index.video_path.iloc[0]).is_file()
    print(split, len(index), "available videos")

## 4. Extract the combined-pruning embeddings

DINOv2 chooses frames before Qwen's visual encoder. DivPrune chooses projected visual tokens before Qwen's first language decoder block. The per-sample DivPrune ratio is computed from the unpruned Qwen visual-token count. Each split is resumable; a failed sample is recorded for later inspection.

In [ ]:
for split in ("dev", "test", "train"):
    output_pt = EMBEDDINGS / f"meld_{split}.pt"
    index_csv = INDEX_DIR / f"meld_{split}_index.csv"
    !python {EXTRACTOR} \
      --index-csv {index_csv} \
      --output-pt {output_pt} \
      --fps 6 \
      --min-frames 4 \
      --max-frames 64 \
      --frame-size 224 \
      --pooling last \
      --prompt-style emotion_task \
      --modality-mode video_text \
      --frame-selection diverse \
      --frame-retain-ratio {FRAME_RETAIN} \
      --frame-encoder-model facebook/dinov2-base \
      --frame-encoder-device cuda \
      --frame-encoder-batch-size 16 \
      --visual-token-pruning divprune \
      --target-combined-visual-retain-ratio {TARGET_VISUAL_RETAIN} \
      --save-dtype float32 \
      --gc-every 5 \
      --batch-save-every 25 \
      --resume

## 5. Verify sample coverage and measured retention

The final ratio below is `retained_visual_token_count / full_visual_token_count`, where the denominator comes from Qwen preprocessing the unpruned frames. This measures the actual combined visual-token retention, rather than multiplying the requested frame and token ratios.

In [ ]:
retention_rows = []
coverage_rows = []
for split in ("train", "dev", "test"):
    payload = torch.load(EMBEDDINGS / f"meld_{split}.pt", map_location="cpu", weights_only=False)
    index = pd.read_csv(INDEX_DIR / f"meld_{split}_index.csv")
    ids = set(payload["sample_ids"])
    error_ids = {row["sample_id"] for row in payload["errors"]}
    assert len(ids) == len(payload["sample_ids"]) == len(payload["metadata"])
    assert not ids & error_ids
    assert ids | error_ids == set(index.sample_id), f"{split}: missing or duplicate sample outcomes"
    assert payload["embeddings"].shape[0] == len(ids) == len(payload["labels"])
    assert torch.isfinite(payload["embeddings"]).all()
    for row in payload["metadata"]:
        assert row["frame_selection"] == "diverse" and row["visual_token_pruning"] == "divprune"
        full = int(row["full_visual_token_count"])
        after_frames = int(row["video_token_count"])
        kept = int(row["retained_visual_token_count"])
        expected = min(after_frames, max(1, round(full * TARGET_VISUAL_RETAIN)))
        assert kept == expected, (row["sample_id"], kept, expected)
        assert row["selected_frame_indices"] == sorted(row["selected_frame_indices"])
        retention_rows.append({
            "split": split, "sample_id": row["sample_id"],
            "candidate_frames": row["candidate_frame_count"],
            "selected_frames": row["retained_frame_count"],
            "full_visual_tokens": full, "tokens_after_frames": after_frames,
            "retained_visual_tokens": kept, "frame_ratio": row["actual_frame_retain_ratio"],
            "post_frame_token_ratio": kept / after_frames,
            "combined_visual_token_ratio": kept / full,
        })
    coverage_rows.append({"split": split, "index_samples": len(index),
                          "embedded": len(ids), "errors": len(error_ids)})
coverage = pd.DataFrame(coverage_rows)
retention = pd.DataFrame(retention_rows)
retention_summary = retention.groupby("split").agg(
    samples=("sample_id", "size"),
    mean_frame_ratio=("frame_ratio", "mean"),
    mean_post_frame_token_ratio=("post_frame_token_ratio", "mean"),
    mean_combined_ratio=("combined_visual_token_ratio", "mean"),
    full_visual_tokens=("full_visual_tokens", "sum"),
    retained_visual_tokens=("retained_visual_tokens", "sum"),
)
retention_summary["weighted_combined_ratio"] = retention_summary.retained_visual_tokens / retention_summary.full_visual_tokens
retention.to_csv(RESULTS / "sample_retention.csv", index=False)
retention_summary.to_csv(RESULTS / "retention_summary.csv")
coverage.to_csv(RESULTS / "coverage.csv", index=False)
display(coverage)
display(retention_summary.round(4))
if coverage.errors.sum():
    print("Some samples failed extraction; inspect the .errors.json files. Metrics will use successful sample IDs.")

## 6. Train a matched emotion MLP

Training uses combined-pruning train embeddings, chooses the checkpoint on dev macro F1, and evaluates on test. This matches the baseline classifier architecture and training settings.

In [ ]:
!python {TRAIN} \
  --train-pt {EMBEDDINGS}/meld_train.pt \
  --dev-pt {EMBEDDINGS}/meld_dev.pt \
  --test-pt {EMBEDDINGS}/meld_test.pt \
  --output-dir {MATCHED} \
  --hidden-dim 512 \
  --dropout 0.3 \
  --epochs 50 \
  --batch-size 64 \
  --learning-rate 3e-4 \
  --weight-decay 1e-4 \
  --seed 42 \
  --device cuda

## 7. Evaluate the unchanged full-baseline MLP

This isolates the representation change from retraining the classifier. The checkpoint and its saved feature normalization stay fixed.

In [ ]:
for split in ("dev", "test"):
    embeddings_pt = EMBEDDINGS / f"meld_{split}.pt"
    !python {EVALUATE} \
      --checkpoint {BASELINE_CHECKPOINT} \
      --embeddings-pt {embeddings_pt} \
      --output-dir {FIXED} \
      --split-name {split} \
      --device cuda

## 8. Compare on identical sample IDs

Macro F1 and accuracy are recomputed on the intersection of baseline, matched-MLP, and fixed-MLP predictions. This avoids claiming a gain merely because an extraction condition skipped different samples.

In [ ]:
comparison_rows = []
for split in ("dev", "test"):
    prediction_paths = {
        "Full baseline": BASELINE / f"outputs/{split}_predictions.csv",
        "Combined pruning + matched MLP": MATCHED / f"{split}_predictions.csv",
        "Combined pruning + fixed baseline MLP": FIXED / f"{split}_predictions.csv",
    }
    frames = {name: pd.read_csv(path) for name, path in prediction_paths.items()}
    shared_ids = set.intersection(*(set(frame.sample_id) for frame in frames.values()))
    assert shared_ids, f"No shared {split} predictions"
    truths = None
    for name, frame in frames.items():
        assert frame.sample_id.is_unique
        aligned = frame[frame.sample_id.isin(shared_ids)].sort_values("sample_id").reset_index(drop=True)
        assert len(aligned) == len(shared_ids)
        if truths is None:
            truths = aligned[["sample_id", "true_id"]]
        else:
            assert aligned[["sample_id", "true_id"]].equals(truths), f"Label mismatch: {name}"
        comparison_rows.append({
            "split": split, "condition": name, "samples": len(shared_ids),
            "accuracy": accuracy_score(aligned.true_id, aligned.pred_id),
            "macro_f1": f1_score(aligned.true_id, aligned.pred_id, labels=list(range(7)), average="macro", zero_division=0),
            "weighted_f1": f1_score(aligned.true_id, aligned.pred_id, labels=list(range(7)), average="weighted", zero_division=0),
        })
    pd.DataFrame({"sample_id": sorted(shared_ids)}).to_csv(RESULTS / f"{split}_shared_sample_ids.csv", index=False)
comparison = pd.DataFrame(comparison_rows)
for split in ("dev", "test"):
    reference = comparison.loc[(comparison.split == split) & (comparison.condition == "Full baseline"), "macro_f1"].iloc[0]
    comparison.loc[comparison.split == split, "delta_macro_f1"] = comparison.loc[comparison.split == split, "macro_f1"] - reference
comparison.to_csv(RESULTS / "aligned_metrics.csv", index=False)
display(comparison.round(4))

## 9. Plot and download results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for split, row in retention_summary.iterrows():
    axes[0].bar(split, row["weighted_combined_ratio"])
axes[0].axhline(TARGET_VISUAL_RETAIN, color="black", linestyle="--", label="15% target")
axes[0].set_ylim(0, max(0.25, retention_summary.weighted_combined_ratio.max() * 1.2))
axes[0].set_ylabel("Retained / unpruned visual tokens")
axes[0].legend()
for condition, part in comparison.groupby("condition"):
    axes[1].plot(part.split, part.macro_f1, marker="o", label=condition)
axes[1].set_ylabel("Macro F1 on shared sample IDs")
axes[1].legend(fontsize=8)
fig.savefig(RESULTS / "retention_and_macro_f1.png", dpi=160)
plt.show()

archive = shutil.make_archive(str(RUN_ROOT) + "_results", "zip", root_dir=RESULTS)
print("Embeddings (resumable):", EMBEDDINGS)
print("Results ZIP:", archive)
display(FileLink(os.path.relpath(archive, Path.cwd())))